# SHIPP — Validate Gold Candidate Matches

**Review draft — read-only.** This notebook validates the persisted Gold table
and the **contract between Gold and the Agent**.

## Why this validator matters
A Gold table can be internally correct and still be unusable by the Agent if
the column names do not match `shipp.agent.contracts.CANDIDATE_MATCH_COLUMNS`.

This notebook checks both:
1. Data-quality / ranking invariants.
2. Agent-read contract compatibility.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
assert table_exists(GOLD_CANDIDATE_MATCHES), (
    f"BLOCKED: {GOLD_CANDIDATE_MATCHES} does not exist. Run Stage 8 first."
)
gold = spark.table(GOLD_CANDIDATE_MATCHES)
print("Gold table:", GOLD_CANDIDATE_MATCHES)
print("Rows:", gold.count())
print("Columns:", gold.columns)
assert gold.count() > 0, "EVIDENCE FAILED: gold_candidate_matches is empty."

In [ ]:
# Import the Agent's actual Gold contract from the repo.
agent_src = os.path.join(REPO_ROOT, "agent", "agent_ship", "src")
if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

from shipp.agent.contracts import CANDIDATE_MATCH_COLUMNS

agent_required = list(CANDIDATE_MATCH_COLUMNS)
missing_for_agent = [c for c in agent_required if c not in gold.columns]

print("Agent requires:", agent_required)
print("Missing from persisted Gold:", missing_for_agent or "none")

# Helpful diagnosis for the current naming pattern.
likely_aliases = {
    "title": "listing_title",
    "condition": "listing_condition",
    "distance_km": "route_distance_km",
    "duration_min": "route_duration_min",
    "computed_at": "gold_processed_at",
}
print("\nLikely rename/alias candidates:")
for agent_col, gold_col in likely_aliases.items():
    if agent_col in missing_for_agent and gold_col in gold.columns:
        print(f"  {agent_col:<15} <- {gold_col}")
if "area" in missing_for_agent:
    print("  area            <- no current Gold column found; requires an explicit privacy-safe area field")

In [ ]:
duplicate_pair = duplicate_key_count(gold, "request_id", "listing_id")
duplicate_rank = duplicate_key_count(gold, "request_id", "match_rank")
score_outside = gold.filter(~F.col("match_score").between(0, 1)).count()
pct_outside = gold.filter(~F.col("match_pct").between(0, 100)).count()
invalid_listing_status = gold.filter(
    ~F.col("listing_status").isin(ELIGIBLE_LISTING_STATUSES)
).count()
distance_without_ok = gold.filter(
    (F.col("route_status") != "OK") & F.col("route_distance_km").isNotNull()
).count()
ok_without_distance = gold.filter(
    (F.col("route_status") == "OK") & F.col("route_distance_km").isNull()
).count()
pending_routes = gold.filter(F.col("route_status") == "PENDING").count()
null_scoring_version = gold.filter(F.col("scoring_version").isNull()).count()

rank_gaps = (
    gold.groupBy("request_id")
    .agg(
        F.count("*").alias("rows"),
        F.min("match_rank").alias("min_rank"),
        F.max("match_rank").alias("max_rank"),
        F.countDistinct("match_rank").alias("distinct_ranks"),
    )
    .filter(
        (F.col("min_rank") != 1)
        | (F.col("max_rank") != F.col("rows"))
        | (F.col("distinct_ranks") != F.col("rows"))
    )
    .count()
)

run_quality_gate({
    "duplicate (request_id, listing_id)": duplicate_pair,
    "duplicate (request_id, match_rank)": duplicate_rank,
    "match_score outside [0,1]": score_outside,
    "match_pct outside [0,100]": pct_outside,
    "listing not eligible": invalid_listing_status,
    "distance present without OK route": distance_without_ok,
    "OK route without distance": ok_without_distance,
    "rank sequence gap": rank_gaps,
    "null scoring_version": null_scoring_version,
    "PENDING route in final Gold evidence": pending_routes,
}, row_count=gold.count(), allow_empty=False)

In [ ]:
display(
    gold.select(
        "request_id", "match_rank", "listing_id",
        "listing_title", "match_score", "match_pct",
        "route_status", "route_distance_km", "route_duration_min",
        "scoring_version", "gold_processed_at",
    )
    .orderBy("request_id", "match_rank")
    .limit(100)
)

In [ ]:
assert not missing_for_agent, (
    "AGENT CONTRACT BLOCKER: gold_candidate_matches cannot be read by the current GoldReader. "
    f"Missing required columns: {missing_for_agent}"
)

print("PASS — Gold quality and Agent-read contract both validated.")

In [ ]:
summary = {
    "stage": "gold_validation",
    "rows": gold.count(),
    "requests_with_matches": gold.select("request_id").distinct().count(),
    "pending_routes": pending_routes,
    "missing_agent_columns": missing_for_agent,
    "status": "PASS",
}
print(json.dumps(summary, indent=2, default=str))